## Step 1 — Load & Inspect

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("dataset/raw_car_dataset.csv")
df.head(10)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['Location'].value_counts(dropna=False)

## Step 2 — Clean Price Column

In [ ]:
df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Price'].dtype, round(df['Price'].skew(), 4)

## Step 3 — Fix Location Errors

In [ ]:
df['Location'] = df['Location'].str.strip().str.title()
df['Location'] = df['Location'].replace({'Subrb': 'Suburb', '??': None})
df['Location'].value_counts(dropna=False)

## Step 4 — Impute Missing Values

In [ ]:
df['Odometer_km'] = df['Odometer_km'].fillna(df['Odometer_km'].median())
df['Doors'] = df['Doors'].fillna(df['Doors'].mode()[0])
df['Accidents'] = df['Accidents'].fillna(df['Accidents'].mode()[0])
df['Location'] = df['Location'].fillna(df['Location'].mode()[0])
df.isnull().sum()

## Step 5 — Remove Duplicates

In [ ]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
f"Removed {before - after} duplicate rows — shape now {df.shape}" 

## Step 6 — IQR Capping (Price & Odometer_km)

In [ ]:
def iqr_cap(col):
    Q1, Q3 = col.quantile(0.25), col.quantile(0.75)
    IQR = Q3 - Q1
    return col.clip(lower=Q1 - 1.5 * IQR, upper=Q3 + 1.5 * IQR)

df['Price'] = iqr_cap(df['Price'])
df['Odometer_km'] = iqr_cap(df['Odometer_km'])
df[['Price', 'Odometer_km']].describe().round(2)

## Step 7 — One-Hot Encode Location

In [ ]:
df = pd.get_dummies(df, columns=['Location'], prefix='Location', dtype=int)
[c for c in df.columns if c.startswith('Location_')]

## Step 8 — Feature Engineering

In [ ]:
df['CarAge'] = 2026 - df['Year']
df['Km_per_year'] = df['Odometer_km'] / df['CarAge'].replace(0, 1)
df['Is_Urban'] = df['Location_City'].astype(int)
df['LogPrice'] = np.log(df['Price'] + 1)
df[['CarAge', 'Km_per_year', 'Is_Urban', 'LogPrice']].head()

## Step 9 — Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['Odometer_km', 'Doors', 'Accidents', 'CarAge', 'Km_per_year']
df[scale_cols] = StandardScaler().fit_transform(df[scale_cols])
df[scale_cols].head()

## Step 10 — Final Checks & Save

In [ ]:
assert df['Price'].dtype == float
assert 'LogPrice' in df.columns
assert df.isnull().sum().sum() == 0
assert any(c.startswith('Location_') for c in df.columns)
print("All checks passed")

In [ ]:
df.isnull().sum()

In [ ]:
df.describe().round(2)

In [ ]:
df.to_csv("car_l3_clean_ready.csv", index=False)
print("Saved.")